# EpiWatch COVID Risk EDA

This notebook explores the processed epidemiological dataset to understand disease trajectory, country-level burden, and how engineered risk signals (Rt, growth, vaccination, policy stringency) interact.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_style('darkgrid')
plt.rcParams['figure.figsize'] = (12, 6)

repo_root = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd()
processed_path = repo_root / 'data' / 'processed.csv'
country_latest_path = repo_root / 'data' / 'country_latest.csv'

processed = pd.read_csv(processed_path)
country_latest = pd.read_csv(country_latest_path)

processed['date'] = pd.to_datetime(processed['date'])
country_latest['date'] = pd.to_datetime(country_latest['date'])

risk_order = ['low', 'moderate', 'high', 'critical']

## Section 1: Dataset Overview

Before drawing conclusions, we inspect the structure and completeness of the dataset. This helps us validate that time coverage, country coverage, and missingness are acceptable for downstream risk interpretation.

In [ ]:
print('Processed shape:', processed.shape)
print('Country latest shape:', country_latest.shape)
print('
Dtypes:
', processed.dtypes)
print('
Null counts:
', processed.isnull().sum().sort_values(ascending=False))
print('
Date range:', processed['date'].min().date(), 'to', processed['date'].max().date())
print('Country count (processed):', processed['country'].nunique())
print('Country count (latest):', country_latest['country'].nunique())

## Section 2: Global Case Trends

We aggregate daily new cases globally to observe broad transmission dynamics. Overlaying a 7-day rolling average reduces weekday reporting noise and highlights underlying waves.

In [ ]:
global_daily = (
    processed.groupby('date', as_index=False)['new_cases']
    .sum()
    .sort_values('date')
)
global_daily['global_7d_avg'] = global_daily['new_cases'].rolling(window=7, min_periods=1).mean()

plt.figure(figsize=(14, 6))
plt.plot(global_daily['date'], global_daily['new_cases'], label='Global Daily New Cases', alpha=0.45)
plt.plot(global_daily['date'], global_daily['global_7d_avg'], label='7-Day Rolling Average', linewidth=2.5)
plt.title('Global COVID-19 Daily Cases with 7-Day Rolling Average')
plt.xlabel('Date')
plt.ylabel('New Cases')
plt.legend()
plt.tight_layout()
plt.show()

## Section 3: Top 20 Countries by Total Cases

Cumulative burden helps identify where transmission has historically concentrated. We rank countries by total confirmed cases and inspect the top 20 for relative scale differences.

In [ ]:
top20 = (
    processed.groupby('country', as_index=False)['confirmed_cases']
    .max()
    .sort_values('confirmed_cases', ascending=False)
    .head(20)
)

plt.figure(figsize=(12, 9))
sns.barplot(data=top20, y='country', x='confirmed_cases', color='#60a5fa')
plt.title('Top 20 Countries by Total Confirmed Cases')
plt.xlabel('Total Confirmed Cases')
plt.ylabel('Country')
plt.tight_layout()
plt.show()

## Section 4: Risk Score Distribution

This distribution shows how current country-level risk is spread across categories. We use the latest country snapshot so each country contributes one point to the current global risk landscape.

In [ ]:
palette = {
    'low': '#22c55e',
    'moderate': '#eab308',
    'high': '#f97316',
    'critical': '#ef4444',
}

plt.figure(figsize=(12, 6))
sns.histplot(
    data=country_latest,
    x='risk_score',
    hue='risk_category',
    hue_order=risk_order,
    multiple='stack',
    bins=30,
    palette=palette,
    edgecolor='white'
)
plt.title('Distribution of Current Country Risk Scores')
plt.xlabel('Risk Score (0-100)')
plt.ylabel('Country Count')
plt.tight_layout()
plt.show()

## Section 5: Rt Distribution by Risk Category

Rt is a core transmission indicator. Comparing Rt across risk categories validates whether the engineered risk classes align with expected epidemic momentum.

In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(
    data=country_latest,
    x='risk_category',
    y='rt_estimate',
    order=risk_order,
    palette=palette
)
plt.title('Rt Estimate Distribution by Risk Category')
plt.xlabel('Risk Category')
plt.ylabel('Rt Estimate')
plt.tight_layout()
plt.show()

## Section 6: Vaccination Coverage vs Risk Score

We examine whether higher vaccination coverage corresponds to lower modeled risk. This relationship can reveal whether immunization levels are reflected in present risk scoring.

In [ ]:
plt.figure(figsize=(12, 7))
sns.scatterplot(
    data=country_latest,
    x='vax_coverage',
    y='risk_score',
    hue='risk_category',
    hue_order=risk_order,
    palette=palette,
    alpha=0.85
)
plt.title('Vaccination Coverage vs Risk Score (Latest Country Snapshot)')
plt.xlabel('People Vaccinated per Hundred')
plt.ylabel('Risk Score')
plt.tight_layout()
plt.show()

## Section 7: Correlation Heatmap

A correlation matrix helps identify linear relationships among numeric variables and flags potential redundancy between engineered features and raw indicators.

In [ ]:
numeric_df = processed.select_dtypes(include=[np.number])
corr = numeric_df.corr(numeric_only=True)

plt.figure(figsize=(14, 10))
sns.heatmap(corr, cmap='coolwarm', annot=True, fmt='.2f', square=False, cbar=True)
plt.title('Correlation Heatmap of Numeric Features')
plt.xlabel('Features')
plt.ylabel('Features')
plt.tight_layout()
plt.show()

## Section 8: Case Trajectory — Top 5 Risk Countries

To understand near-term epidemic behavior, we focus on the five highest-risk countries and plot their recent 90-day case trajectories. This compares pace and volatility across the most concerning settings.

In [ ]:
top5_risk = (
    country_latest.sort_values('risk_score', ascending=False)
    .head(5)['country']
    .tolist()
)

latest_date = processed['date'].max()
start_date = latest_date - pd.Timedelta(days=90)
trajectory = processed[
    (processed['country'].isin(top5_risk)) &
    (processed['date'] >= start_date)
].copy()

plt.figure(figsize=(14, 7))
for country in top5_risk:
    subset = trajectory[trajectory['country'] == country].sort_values('date')
    plt.plot(subset['date'], subset['new_cases_smoothed'], label=country, linewidth=2)

plt.title('Last 90 Days: New Cases Smoothed for Top 5 Risk Countries')
plt.xlabel('Date')
plt.ylabel('New Cases Smoothed')
plt.legend(title='Country')
plt.tight_layout()
plt.show()